# SQL Agents and Data Tools

This notebook collects practical patterns for letting an agent work with tables safely.

The goal is not to let the model freestyle SQL. The goal is to combine schema awareness, tool boundaries, validation, and row limits so the system can answer real business questions without being risky.


## What to optimize for

Useful habits for enterprise SQL agents:

- inspect schema before writing a query
- prefer read-only tools
- enforce row limits and timeouts
- validate generated SQL before execution
- expose a human-readable query plan or explanation
- keep the model away from direct credentials


In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class QueryIntent:
    question: str
    tables: list[str]
    max_rows: int = 100

intent = QueryIntent(
    question='What are the top 5 product categories by revenue this month?',
    tables=['orders', 'products', 'revenue'],
    max_rows=5,
)

intent


In [ ]:
SCHEMA = {
    'orders': ['order_id', 'customer_id', 'product_id', 'order_date', 'amount'],
    'products': ['product_id', 'category', 'region'],
}

def describe_schema(schema: dict[str, list[str]]) -> str:
    lines = []
    for table, columns in schema.items():
        lines.append(f'{table}: {', '.join(columns)}')
    return '\n'.join(lines)

print(describe_schema(SCHEMA))


## Safe query generation pattern

Instead of asking the model to output raw SQL immediately, ask it for a structured intent first. Then render SQL from a template or validator layer.


In [ ]:
def build_safe_sql(intent: QueryIntent) -> str:
    base = "SELECT product_id, SUM(amount) AS revenue FROM orders"
    where = " WHERE order_date >= DATE('now', '-30 day')"
    group_by = " GROUP BY product_id ORDER BY revenue DESC"
    limit = f" LIMIT {intent.max_rows}"
    return base + where + group_by + limit

build_safe_sql(intent)


In [ ]:
def validate_sql(sql: str) -> list[str]:
    warnings = []
    lowered = sql.lower()
    if not lowered.startswith('select'):
        warnings.append('Only SELECT queries are allowed.')
    if 'limit' not in lowered:
        warnings.append('Add a row limit for safety.')
    if any(word in lowered for word in ['delete', 'update', 'drop', 'insert', 'alter']):
        warnings.append('Mutation keywords are blocked.')
    return warnings

query = build_safe_sql(intent)
{'sql': query, 'warnings': validate_sql(query)}


## Tool design tips

When you expose SQL tools to an agent, keep each tool narrow:

- `list_tables`
- `describe_table`
- `run_readonly_sql`
- `sample_rows`
- `explain_result`

That lets LangChain or LangGraph reason step by step without giving the model too much freedom.


In [ ]:
def sample_rows(table: str) -> list[dict[str, Any]]:
    if table == 'orders':
        return [
            {'order_id': 1, 'customer_id': 101, 'product_id': 501, 'amount': 120.0},
            {'order_id': 2, 'customer_id': 102, 'product_id': 502, 'amount': 89.0},
        ]
    return []

sample_rows('orders')


## LangChain and LangGraph usage notes

A current pattern is:

1. infer intent
2. inspect schema
3. generate query
4. validate query
5. execute through a read-only tool
6. summarize the result with citations to the query and tables used

If the query is ambiguous, route back to clarification instead of guessing.


## Production reminders

- cache schema metadata where safe
- log the SQL and execution time
- reject expensive queries
- paginate large results
- keep secrets in the database layer, not in prompts
- use local or hosted LLMs only for orchestration, not direct data access


## Demo ideas

Good interview scenarios for this notebook:

- revenue by region and category
- churn-risk cohorts by month
- top incident types by team
- SLA breach analysis with safe filters
